# A text-to-query workflow in practice

The end-to-end text-to-query workflow using MongoDB involves the following steps:

![](images/text_to_query.png)

1. Identify the collection to query
2. Examine the schema of the collection to query
3. Generate the MongoDB query
4. Validate the MongoDB query
5. Execute the MongoDB query
6. Reason over the results obtained

In this exercise, we will manually do step 1 through 6. For Step 6, we will pass the results from executing the query to an LLM to reason over them and generate a natural language response to the question.

For simplicity, let's use one of the sample queries from the previous exercise. We already know that the data for this query is present in the `movies` collection, we have already previewed the schema of a sample document, and also created the query. So let's continue the text-to-query workflow from Step 4.

**Re-run the cells below to install `pymongo` and access the `"movies"` collection.**

In [8]:
!pip install  --quiet pymongo==4.13.2


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [9]:
import os
from pymongo import MongoClient

MONGODB_URI = os.environ["MONGODB_URI"]

# Initialize a MongoDB Python client
mongodb_client = MongoClient(MONGODB_URI)

# Access the sample_mflix database
db = mongodb_client["sample_mflix"]

# Access the movies collection
collection = db["movies"]

**Initialize an OpenAI LLM using LangChain's `ChatOpenAI` class.**

In [10]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

**Complete the code below to convert a natural language query to a MongoDB query, execute it and obtain the results as a Python list.**

In [11]:
# Natural language user query
user_query = "Give me the top 5 directors by average IMDB rating, who have made at least 20 movies"

# Convert natural language query to MongoDB query
mongodb_query = [{"$unwind": "$directors"},
          {"$group": {"_id": "$directors", "filmCount": {"$sum": 1}, "avgRating": {"$avg": "$imdb.rating"}}},
          {"$match": {"filmCount": {"$gte": 20}}},
          {"$sort": {"avgRating": -1}},
          {"$limit": 5}]

# Execute the query and obtain results
docs = [doc for doc in collection.aggregate(mongodb_query)]
docs

[{'_id': 'William Wyler', 'filmCount': 21, 'avgRating': 7.676190476190476},
 {'_id': 'Martin Scorsese', 'filmCount': 32, 'avgRating': 7.640625},
 {'_id': 'Alfred Hitchcock', 'filmCount': 24, 'avgRating': 7.5874999999999995},
 {'_id': 'Steven Spielberg', 'filmCount': 29, 'avgRating': 7.479310344827587},
 {'_id': 'Woody Allen', 'filmCount': 40, 'avgRating': 7.215000000000001}]

For Step 6, we will first create a prompt for the LLM using the `ChatPromptTemplate` class in LangChain. This allows you to create flexible templated prompts.

Our chat prompt template will consist of a system prompt with a placeholder for the query results (`context`), and a placeholder for user and AI messages(`MessagesPlaceholder`).

We will use the `.from_messages()` method of the `ChatPromptTemplate` class since we are using different message formats, such as a 2-tuple of (message type, template) and `MessagesPlaceholder`, to create the prompt.

**Create a templated prompt for the LLM, consisting of a system prompt and a placeholder for messages, using the `.from_messages()` method.**

In [12]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Create a system prompt for the LLM
system_prompt = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Context: {context}"""

# Create a templated prompt for the LLM
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

To pass the prompt to the LLM, we will use the `|` operator, which in LangChain, allows you to "chain" prompts, LLMs, parsers etc. sequentially, with the output of one serving as the input to the next.

**Chain the prompt with the LLM using the `|` operator.**

In [13]:
# Chain the prompt with the LLM
llm_with_prompt = prompt | llm

You can call LLMs, tools etc. in LangChain using the `.invoke()` method. The inputs of the `.invoke()` method are the inputs to the LLM/tool. You can also pass placeholder values as inputs to the `.invoke()` method.

**Invoke the `llm_with_prompt` chain with the query results and the user query as inputs.**

In [14]:
import json

response = llm_with_prompt.invoke({"context": json.dumps(docs),
                                   "messages": [("user", user_query)]})
# Print the LLM's response
print(response.content, end="")

The top 5 directors by average IMDB rating, who have made at least 20 movies, are:

1. William Wyler - Avg Rating: 7.68 (Film Count: 21)
2. Martin Scorsese - Avg Rating: 7.64 (Film Count: 32)
3. Alfred Hitchcock - Avg Rating: 7.59 (Film Count: 24)
4. Steven Spielberg - Avg Rating: 7.48 (Film Count: 29)
5. Woody Allen - Avg Rating: 7.22 (Film Count: 40)